# COMPASS univariate models

Per-landmark univariate Cox arm, plus a nominal-significance filter over
those results. Requires `01_preprocessing.ipynb` to have built the merged
`profile_data` inputs under `prediction_inputs_<arm>/` first.

In [ ]:
ARMS = ["adt"]
# Which event to model. "platinum" reproduces the original run exactly.
# "nepc" models time from the ADT anchor to the LLM-adjudicated NEPC
# diagnosis; it needs OUTPUT_SUFFIX = "_nepc" so it reads/writes its own
# prediction_inputs_adt_nepc/ and local_runs_adt_nepc/ trees and leaves the
# platinum results untouched. The two cohorts are NOT the same patients: the
# NEPC build is gated on t_nepc > 0 (incident NEPC only).
ENDPOINT = "platinum"
OUTPUT_SUFFIX = ""  # "_nepc" when ENDPOINT == "nepc"
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.ENDPOINT = ENDPOINT
cp.FORCE_RERUN = OVERWRITE

RUNS = cp.make_runs(ARMS, output_suffix=OUTPUT_SUFFIX)

## Run univariate models

Per-landmark univariate arm. Set `OVERWRITE = True` in the configuration cell to refit and replace existing outputs. With `False`, completed landmarks are skipped.

In [ ]:
for run in RUNS:
    cp.run_univariate(run)

## Nominally significant results

Filters the per-landmark univariate results to `p_value < 0.05` (nominal,
not multiplicity-adjusted -- `q_value` is retained in the export for that) and
writes per-run tables to `cox/nominally_significant_univariate_results.csv`
beneath the `profile_data` output root.

In [ ]:
NOMINAL_ALPHA = 0.05

nominal_tables = {}
for run in RUNS:
    results = cp.load_univariate_results(run)
    filtered = cp.filter_nominal(results, alpha=NOMINAL_ALPHA)
    nominal_tables[run["label"]] = filtered

    export_path = run["output_dir"] / "cox" / "nominally_significant_univariate_results.csv"
    if export_path.exists() and not OVERWRITE:
        print(f"{run['label']}: keeping existing nominal-significance export -> {export_path}")
    else:
        export_path.parent.mkdir(parents=True, exist_ok=True)
        filtered.to_csv(export_path, index=False)
        print(f"{run['label']}: {len(filtered)} nominally significant rows -> {export_path}")

In [ ]:
nominal_tables[RUNS[0]["label"]]

## Separate sequencing, Gleason, and PRS univariate runs

Runs three distinct Cox analyses: somatic alterations from the sequencing sample closest to ADT start, followed from specimen collection to the endpoint; the Gleason score closest to ADT start, followed from the score date to the endpoint; and PRSs followed from ADT start to the endpoint. Results are written beneath `cox_somatic_gleason/`, separate from the lab associations.

These follow `ENDPOINT` like the lab arm above, but they read the separate `somatic_gleason/` input build. Under `ENDPOINT = "nepc"` that build must have been produced from the NEPC prediction-inputs tree in `01_preprocessing.ipynb`; otherwise skip these cells and compare the lab arm only.

In [ ]:
for run in RUNS:
    cp.run_somatic_gleason_univariate(run)

In [ ]:
somatic_gleason_tables = {}
for run in RUNS:
    results = cp.load_somatic_gleason_univariate_results(run)
    filtered = cp.filter_nominal(results, alpha=NOMINAL_ALPHA)
    somatic_gleason_tables[run["label"]] = filtered
    export_path = run["output_dir"] / "cox_somatic_gleason" / "nominally_significant_univariate_results.csv"
    if export_path.exists() and not OVERWRITE:
        print(f"{run['label']}: keeping existing nominal sequencing/Gleason/PRS export -> {export_path}")
    else:
        export_path.parent.mkdir(parents=True, exist_ok=True)
        filtered.to_csv(export_path, index=False)
        print(f"{run['label']}: {len(filtered)} nominal sequencing/Gleason/PRS rows -> {export_path}")

In [ ]:
somatic_gleason_tables[RUNS[0]["label"]]